# LeetCode 3423 — Maximum Difference Between Adjacent Elements in a Circular Array bằng Python

Link đề: https://leetcode.com/problems/maximum-difference-between-adjacent-elements-in-a-circular-array/ · Độ khó: **Easy** · Chủ đề: Array (mảng), Circular Array (mảng vòng)

Skeleton LeetCode:

```python
class Solution:
    def maxAdjacentDistance(self, nums: List[int]) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (so sánh nhiều cách làm với nhau và với vét cạn theo định nghĩa). Ví dụ nào do mình tự dựng sẽ ghi rõ "tự dựng" — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng `nums` là mảng **vòng** (phần tử cuối **kề** với phần tử đầu); cần tìm **giá trị tuyệt đối lớn nhất** của hiệu hai phần tử **kề nhau**.
2. Nắm các công cụ Python: chỉ số **âm** (`nums[-1]`), phép **chia dư** `%` để quay vòng chỉ số, `abs`, `max` kèm biểu thức sinh (generator), `zip` và cắt lát `nums[1:]`, `itertools.pairwise`.
3. Đi từ cách **vét cạn mọi cặp rồi lọc cặp kề** → **chỉ xét đúng `n` cặp kề** bằng `(i + 1) % n` → **mẹo chỉ số âm `nums[i - 1]`** → **thêm phần tử đầu vào cuối rồi duyệt cặp liền kề** → **một lượt duyệt, cuối cùng so thêm cặp (cuối, đầu)**.
4. Hiểu mấu chốt: mảng vòng gồm đúng **`n` cặp kề** (`n - 1` cặp thường cộng **một cặp "vòng"** nối cuối với đầu). Ta không cần xét mọi `n·(n−1)/2` cặp.
5. Tránh các lỗi kinh điển: quên cặp vòng (chỉ chạy `range(n - 1)`), `IndexError` khi viết `nums[i + 1]` ở phần tử cuối, quên `abs`, và nhầm bài này với `max(nums) - min(nums)`.

## 2. Đề bài & ví dụ

Cho một mảng **vòng** `nums`. Hãy tìm **hiệu tuyệt đối lớn nhất** giữa hai phần tử **kề nhau**.

**Lưu ý**: trong mảng vòng, phần tử **đầu** và phần tử **cuối** được coi là **kề nhau**.

**Ví dụ 1**

- Input: `nums = [1,2,4]`
- Output: `3`
- Giải thích: vì `nums` là mảng vòng nên `nums[0]` và `nums[2]` kề nhau; chúng có hiệu tuyệt đối lớn nhất là `|4 - 1| = 3`.

**Ví dụ 2**

- Input: `nums = [-5,-10,-5]`
- Output: `5`
- Giải thích: hai phần tử kề `nums[0]` và `nums[1]` có hiệu tuyệt đối lớn nhất là `|-5 - (-10)| = 5`.

**Ràng buộc**

- `2 <= nums.length <= 100`
- `-100 <= nums[i] <= 100`

In [1]:
from typing import List

examples = [
    ([1, 2, 4], 3),
    ([-5, -10, -5], 5),
]

for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 4] -> kỳ vọng 3
nums = [-5, -10, -5] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Mảng vòng nghĩa là gì?

Hãy tưởng tượng các phần tử xếp quanh một chiếc **bàn tròn**: người ngồi cuối bàn **kề** người ngồi đầu bàn. Với `nums = [1, 2, 4]` ta có ba cặp kề: `(1, 2)`, `(2, 4)` và `(4, 1)`. Tổng quát, mảng có `n` phần tử vòng có **đúng `n` cặp kề** (khi `n >= 3`). Khi `n = 2` thì hai cặp `(a, b)` và `(b, a)` trùng nhau về giá trị tuyệt đối.

In [2]:
nums = [1, 2, 4]
n = len(nums)
for i in range(n):
    j = (i + 1) % n
    print(f"cặp kề thứ {i}: nums[{i}]={nums[i]} và nums[{j}]={nums[j]}")

cặp kề thứ 0: nums[0]=1 và nums[1]=2
cặp kề thứ 1: nums[1]=2 và nums[2]=4
cặp kề thứ 2: nums[2]=4 và nums[0]=1


### 3.2. Phép chia dư `%` để "quay vòng" chỉ số

Với `n` phần tử, chỉ số hợp lệ là `0..n-1`. Biểu thức `(i + 1) % n` cho chỉ số của **phần tử kế tiếp**; khi `i = n - 1` thì `(n - 1 + 1) % n = n % n = 0`, tức quay về đầu. Đây là mẹo kinh điển cho mọi bài về **mảng vòng**.

In [3]:
n = 5
for i in range(n):
    print(f"i={i} -> kế tiếp (i+1)%n = {(i + 1) % n}, lùi (i-1)%n = {(i - 1) % n}")

i=0 -> kế tiếp (i+1)%n = 1, lùi (i-1)%n = 4
i=1 -> kế tiếp (i+1)%n = 2, lùi (i-1)%n = 0
i=2 -> kế tiếp (i+1)%n = 3, lùi (i-1)%n = 1
i=3 -> kế tiếp (i+1)%n = 4, lùi (i-1)%n = 2
i=4 -> kế tiếp (i+1)%n = 0, lùi (i-1)%n = 3


### 3.3. Chỉ số âm trong Python: `nums[-1]`

Python cho phép chỉ số **âm**: `nums[-1]` là phần tử **cuối**, `nums[-2]` là phần tử kế cuối. Vì vậy `nums[i - 1]` khi `i = 0` chính là `nums[-1]` — phần tử cuối — đúng thứ ta cần cho cặp vòng! Đây là đặc sản của Python (nhiều ngôn ngữ khác sẽ báo lỗi).

In [4]:
nums = [10, 20, 30, 40]
print(nums[-1], nums[-2])                     # 40 30
for i in range(len(nums)):
    print(f"i={i}: nums[i]={nums[i]}, nums[i-1]={nums[i - 1]}")      # i=0 -> nums[-1] = 40

40 30
i=0: nums[i]=10, nums[i-1]=40
i=1: nums[i]=20, nums[i-1]=10
i=2: nums[i]=30, nums[i-1]=20
i=3: nums[i]=40, nums[i-1]=30


### 3.4. `abs` và `max` với biểu thức sinh

`abs(x)` là giá trị tuyệt đối (`abs(-5) = 5`). `max(biểu_thức for ... in ...)` tìm giá trị lớn nhất của một dãy sinh ra tại chỗ, **không cần tạo list**. Ví dụ `max(abs(a - b) for ...)`.

In [5]:
print(abs(-5), abs(5), abs(0))                                      # 5 5 0
print(max(abs(x - 3) for x in [1, 2, 4, 9]))                        # |9-3| = 6
print(max([3, 1, 2]), max(3, 7))                                    # 3 7

5 5 0
6
3 7


### 3.5. `zip` và cắt lát `nums[1:]`

`zip(a, b)` ghép từng cặp phần tử cùng vị trí của hai dãy và **dừng ở dãy ngắn hơn**. `nums[1:]` là nums bỏ phần tử đầu. Vậy `zip(nums, nums[1:])` cho ra các cặp **liền kề** `(nums[0], nums[1]), (nums[1], nums[2]), ...` — đúng `n − 1` cặp. Muốn thêm cặp vòng, ghép với `nums[1:] + nums[:1]` (xoay trái một bước).

In [6]:
nums = [1, 2, 4]
print(list(zip(nums, nums[1:])))                    # [(1, 2), (2, 4)]  -> thiếu cặp vòng
rotated = nums[1:] + nums[:1]                       # [2, 4, 1]
print(rotated)
print(list(zip(nums, rotated)))                     # [(1, 2), (2, 4), (4, 1)]  -> đủ 3 cặp

[(1, 2), (2, 4)]
[2, 4, 1]
[(1, 2), (2, 4), (4, 1)]


### 3.6. `itertools.pairwise` (Python 3.10+)

`pairwise(a)` trả về các cặp liền kề `(a0, a1), (a1, a2), ...` — giống `zip(a, a[1:])` nhưng gọn hơn. Nếu thêm `nums[0]` vào **cuối** danh sách rồi dùng `pairwise`, ta được luôn cặp vòng `(nums[-1], nums[0])`.

In [7]:
from itertools import pairwise

nums = [1, 2, 4]
print(list(pairwise(nums)))                         # [(1, 2), (2, 4)]
print(list(pairwise(nums + [nums[0]])))             # [(1, 2), (2, 4), (4, 1)]

[(1, 2), (2, 4)]
[(1, 2), (2, 4), (4, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [1, 2, 4]`. Liệt kê **mọi cặp kề** theo vòng tròn rồi lấy hiệu tuyệt đối:

| cặp kề | giá trị | `|hiệu|` |
|---|---|---|
| `nums[0]`, `nums[1]` | `1`, `2` | 1 |
| `nums[1]`, `nums[2]` | `2`, `4` | 2 |
| `nums[2]`, `nums[0]` (cặp vòng) | `4`, `1` | 3 |

Giá trị lớn nhất là `3`, đúng đáp án. Nếu **quên cặp vòng**, ta chỉ thấy `max(1, 2) = 2` — **sai**. Vậy bài này chỉ có hai việc: (1) liệt kê **đủ** các cặp kề, kể cả cặp vòng; (2) lấy `max` của `|hiệu|`.

In [8]:
nums = [1, 2, 4]
n = len(nums)
best = 0
for i in range(n):
    a, b = nums[i], nums[(i + 1) % n]
    d = abs(a - b)
    best = max(best, d)
    print(f"|{a} - {b}| = {d}, best = {best}")

|1 - 2| = 1, best = 1
|2 - 4| = 2, best = 2
|4 - 1| = 3, best = 3


## 5. Cách 1 — Brute Force / baseline: xét mọi cặp rồi lọc cặp kề

**Ý tưởng.** Làm đúng theo định nghĩa "chỉ xét cặp kề":

1. Duyệt mọi cặp chỉ số `i < j`.
2. Hai phần tử `nums[i]`, `nums[j]` **kề nhau trong mảng vòng** khi khoảng cách chỉ số `j - i` bằng `1` (kề thường) **hoặc** bằng `n - 1` (cặp vòng: `i = 0`, `j = n - 1`).
3. Với cặp kề, cập nhật `best = max(best, abs(nums[i] - nums[j]))`.

Cách này **chậm hơn mức cần thiết** nhưng bám sát định nghĩa, và sau này ta dùng nó làm "trọng tài" để kiểm tra các cách nhanh.

In [9]:
def max_adjacent_all_pairs(nums: List[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            gap = j - i
            if gap == 1 or gap == n - 1:             # kề thường hoặc cặp vòng
                best = max(best, abs(nums[i] - nums[j]))
    return best


for nums, expected in examples:
    print(max_adjacent_all_pairs(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


**Độ phức tạp (Cách 1).** Hai vòng lồng nhau ⇒ **Time `O(n²)`**, **Space `O(1)`**. Với `n <= 100` là khoảng `5000` phép kiểm tra — vẫn nhanh, nhưng ta kiểm tra rất nhiều cặp **không** kề.

**Nút thắt:** chỉ có **`n` cặp kề**, mà ta lại duyệt `n(n−1)/2` cặp. Với mỗi `i`, cặp kề tiếp theo là **biết trước** (`(i + 1) % n`) nên không cần vòng lặp trong.

## 6. Ý tưởng thuật toán / bước đột phá

**Đi thẳng đến cặp kề.** Với mỗi vị trí `i` từ `0` đến `n - 1`, phần tử kề phía sau là `nums[(i + 1) % n]`. Chỉ `n` cặp, mỗi cặp tính trong `O(1)` ⇒ `O(n)` tổng cộng. Python còn cho mẹo ngắn hơn: `nums[i - 1]` (kề phía **trước**, với `i = 0` thì rơi vào `nums[-1]`), và cặp `(nums[i - 1], nums[i])` với `i = 0..n-1` cũng chính là `n` cặp kề của vòng tròn.

Hai cách liệt kê (kề **sau** và kề **trước**) cho **cùng một tập cặp**, chỉ khác thứ tự trong cặp — mà `abs` không quan tâm thứ tự. Cell dưới đây kiểm chứng điều đó trên một ví dụ.

In [10]:
nums = [1, 2, 4]
n = len(nums)
after = [(nums[i], nums[(i + 1) % n]) for i in range(n)]
before = [(nums[i - 1], nums[i]) for i in range(n)]
print("kề sau :", after)
print("kề trước:", before)
print("cùng tập cặp (bỏ qua thứ tự):",
      sorted(map(sorted, after)) == sorted(map(sorted, before)))

kề sau : [(1, 2), (2, 4), (4, 1)]
kề trước: [(4, 1), (1, 2), (2, 4)]
cùng tập cặp (bỏ qua thứ tự): True


## 7. Các cách tiếp theo

### Cách 2 — Vòng lặp với `(i + 1) % n`

**Giải thích từng đoạn.**

1. `n = len(nums)` và `best = 0` (hiệu tuyệt đối không bao giờ âm nên `0` là giá trị khởi đầu an toàn; vì `n >= 2` luôn có ít nhất một cặp).
2. `for i in range(n)`: mỗi `i` đại diện cho cặp `(nums[i], nums[(i + 1) % n])`.
3. Cập nhật `best = max(best, abs(...))`.

In [11]:
def max_adjacent_modulo(nums: List[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        d = abs(nums[i] - nums[(i + 1) % n])          # (i + 1) % n quay vòng về 0 khi i = n - 1
        if d > best:
            best = d
    return best


for nums, expected in examples:
    print(max_adjacent_modulo(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionModulo:
    def maxAdjacentDistance(self, nums: List[int]) -> int:
        n = len(nums)
        best = 0
        for i in range(n):
            best = max(best, abs(nums[i] - nums[(i + 1) % n]))
        return best


print(SolutionModulo().maxAdjacentDistance([1, 2, 4]))          # 3

3


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Mẹo chỉ số âm `nums[i - 1]`

**Giải thích.** Với `i` chạy từ `0` đến `n - 1`, cặp `(nums[i - 1], nums[i])` cho đủ `n` cặp kề; riêng `i = 0` Python tự hiểu `nums[-1]` là phần tử cuối nên ta có cặp vòng **miễn phí**. Gọn nhưng chỉ chạy được trong Python (và vài ngôn ngữ có chỉ số âm); khi chuyển sang C++/Java phải dùng `%`.

In [13]:
def max_adjacent_negative_index(nums: List[int]) -> int:
    return max(abs(nums[i] - nums[i - 1]) for i in range(len(nums)))      # i = 0 -> nums[-1]


for nums, expected in examples:
    print(max_adjacent_negative_index(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionNegativeIndex:
    def maxAdjacentDistance(self, nums: List[int]) -> int:
        return max(abs(nums[i] - nums[i - 1]) for i in range(len(nums)))


print(SolutionNegativeIndex().maxAdjacentDistance([-5, -10, -5]))       # 5

5


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — Thêm phần tử đầu vào cuối rồi `pairwise`

**Giải thích.** `nums + [nums[0]]` làm cho cặp vòng trở thành một cặp liền kề thông thường `(nums[-1], nums[0])` ở cuối. Sau đó `pairwise` liệt kê các cặp liền kề và `max(abs(a - b) ...)` chọn kết quả. Đây là cách trong lời giải mẫu của đề.

In [15]:
from itertools import pairwise


def max_adjacent_pairwise(nums: List[int]) -> int:
    extended = nums + [nums[0]]                         # [1, 2, 4] -> [1, 2, 4, 1]
    return max(abs(a - b) for a, b in pairwise(extended))


for nums, expected in examples:
    print(max_adjacent_pairwise(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


In [16]:
# Code nộp LeetCode cho Cách 4
from itertools import pairwise


class SolutionPairwise:
    def maxAdjacentDistance(self, nums: List[int]) -> int:
        return max(abs(a - b) for a, b in pairwise(nums + [nums[0]]))


print(SolutionPairwise().maxAdjacentDistance([1, 2, 4]))        # 3

3


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (tạo danh sách mới `nums + [nums[0]]`).

### Cách 5 — Một lượt duyệt, cuối cùng so thêm cặp (cuối, đầu)

**Giải thích.** Tách bài thành hai phần: (1) duyệt `n - 1` cặp liền kề thông thường `(nums[i], nums[i + 1])` với `i` từ `0` đến `n - 2`; (2) **sau vòng lặp**, so thêm cặp vòng `abs(nums[-1] - nums[0])`. Cách này không cần `%`, không cần chỉ số âm, không tạo list phụ — dễ chuyển sang ngôn ngữ khác nhất.

In [17]:
def max_adjacent_scan_then_wrap(nums: List[int]) -> int:
    best = 0
    for i in range(len(nums) - 1):                      # n - 1 cặp liền kề
        best = max(best, abs(nums[i] - nums[i + 1]))
    best = max(best, abs(nums[-1] - nums[0]))           # cặp vòng
    return best


for nums, expected in examples:
    print(max_adjacent_scan_then_wrap(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionScanThenWrap:
    def maxAdjacentDistance(self, nums: List[int]) -> int:
        best = abs(nums[-1] - nums[0])                  # bắt đầu bằng cặp vòng
        for i in range(len(nums) - 1):
            best = max(best, abs(nums[i] - nums[i + 1]))
        return best


print(SolutionScanThenWrap().maxAdjacentDistance([1, 2, 4]))    # 3

3


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

Cả bốn cách sau đều `O(n)`. Bản nộp mình chọn là **Cách 2** (`(i + 1) % n`): dùng đúng công cụ "quay vòng chỉ số" mà bạn sẽ gặp lại ở nhiều bài mảng vòng khác.

## 8. TRACE từng bước

Trace **Cách 2** (`(i + 1) % n`) với `nums = [-5, -10, -5]`:

| `i` | `nums[i]` | `(i+1) % n` | `nums[(i+1) % n]` | `|hiệu|` | `best` sau |
|---|---|---|---|---|---|
| 0 | -5 | 1 | -10 | 5 | 5 |
| 1 | -10 | 2 | -5 | 5 | 5 |
| 2 | -5 | 0 | -5 | 0 | 5 |

Hàng cuối (`i = 2`) là **cặp vòng**: `nums[2]` kề với `nums[0]`; hiệu `|-5 - (-5)| = 0` không làm đổi `best`.

In [19]:
def trace_modulo(nums: List[int]) -> int:
    n = len(nums)
    best = 0
    print(f"nums = {nums}, n = {n}")
    for i in range(n):
        j = (i + 1) % n
        d = abs(nums[i] - nums[j])
        best = max(best, d)
        wrap = "  <- cặp vòng" if j == 0 else ""
        print(f"  i={i} j={j}: |{nums[i]} - {nums[j]}| = {d}, best = {best}{wrap}")
    return best


assert trace_modulo([-5, -10, -5]) == 5

nums = [-5, -10, -5], n = 3
  i=0 j=1: |-5 - -10| = 5, best = 5
  i=1 j=2: |-10 - -5| = 5, best = 5
  i=2 j=0: |-5 - -5| = 0, best = 5  <- cặp vòng


Trace **Cách 5** (quét `n − 1` cặp rồi cặp vòng) với `nums = [1, 2, 4]` — để thấy rõ cặp vòng mới là cặp cho đáp án.

In [20]:
def trace_scan_then_wrap(nums: List[int]) -> int:
    best = 0
    print(f"nums = {nums}")
    for i in range(len(nums) - 1):
        d = abs(nums[i] - nums[i + 1])
        best = max(best, d)
        print(f"  cặp liền kề i={i}: |{nums[i]} - {nums[i + 1]}| = {d}, best = {best}")
    d = abs(nums[-1] - nums[0])
    best = max(best, d)
    print(f"  cặp vòng: |{nums[-1]} - {nums[0]}| = {d}, best = {best}")
    return best


assert trace_scan_then_wrap([1, 2, 4]) == 3

nums = [1, 2, 4]
  cặp liền kề i=0: |1 - 2| = 1, best = 1
  cặp liền kề i=1: |2 - 4| = 2, best = 2
  cặp vòng: |4 - 1| = 3, best = 3


## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (`(i + 1) % n`). `List` đã được LeetCode import sẵn; ở notebook mình đã `from typing import List`.

In [21]:
class Solution:
    def maxAdjacentDistance(self, nums: List[int]) -> int:
        n = len(nums)
        best = 0
        for i in range(n):
            # (i + 1) % n: khi i là phần tử cuối thì quay về phần tử đầu (cặp vòng)
            best = max(best, abs(nums[i] - nums[(i + 1) % n]))
        return best


sol = Solution()
for nums, expected in examples:
    print(sol.maxAdjacentDistance(nums), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


## 10. Vì sao thuật toán đúng

- **Đủ cặp kề:** mảng vòng `n` phần tử có các cặp kề `{i, (i + 1) mod n}` với `i = 0..n-1`. Vòng lặp `for i in range(n)` đi qua đúng các cặp đó; khi `i = n - 1` ta được cặp `(nums[n-1], nums[0])`, tức cặp vòng.
- **Không thừa:** không cặp nào ngoài `n` cặp này là kề nhau, nên `max` trên các cặp này chính là đáp án.
- **`abs` đúng nghĩa:** đề hỏi **hiệu tuyệt đối**, nên thứ tự trong cặp không quan trọng.
- **Khởi tạo `best = 0`:** hiệu tuyệt đối luôn `>= 0`; và vì `n >= 2` luôn có ít nhất một cặp, `best` sẽ được cập nhật theo đúng cặp lớn nhất (nếu mọi hiệu bằng `0` thì đáp án đúng là `0`).
- **`n = 2`:** hai cặp `(a, b)` và `(b, a)` cho cùng `|a - b|`, nên không có vấn đề trùng lặp.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp rồi lọc kề | `O(n²)` | `O(1)` |
| 2. `(i + 1) % n` | `O(n)` | `O(1)` |
| 3. Chỉ số âm `nums[i - 1]` | `O(n)` | `O(1)` |
| 4. `nums + [nums[0]]` + `pairwise` | `O(n)` | `O(n)` |
| 5. Quét `n − 1` cặp + cặp vòng | `O(n)` | `O(1)` |

Không thể tốt hơn `O(n)` vì phải nhìn mọi phần tử ít nhất một lần.

## 12. Bộ test

Gom các cách vào `impls` rồi so với kỳ vọng. "Trọng tài" là Cách 1 (mọi cặp, lọc theo định nghĩa) và thêm một hàm kiểm tra viết riêng dùng **khoảng cách vòng** `min(d, n - d)`.

In [22]:
import random

sol = Solution()
impls = {
    "all_pairs": max_adjacent_all_pairs,
    "modulo": max_adjacent_modulo,
    "negative_index": max_adjacent_negative_index,
    "pairwise": max_adjacent_pairwise,
    "scan_then_wrap": max_adjacent_scan_then_wrap,
    "leetcode": sol.maxAdjacentDistance,
}


def oracle_circular_distance(nums):
    """Trọng tài: hai chỉ số kề nhau khi khoảng cách vòng min(d, n-d) == 1."""
    n = len(nums)
    best = 0
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            d = abs(i - j)
            if min(d, n - d) == 1:
                best = max(best, abs(nums[i] - nums[j]))
    return best


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
    assert oracle_circular_distance(nums) == expected, (label, "oracle")
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Các ví dụ "tự dựng" bên dưới có kỳ vọng được tính tay ngay trong comment.

In [23]:
check_all([5, 5], 0, "hai phần tử bằng nhau")
check_all([1, 100], 99, "hai phần tử: |1-100|")
check_all([-100, 100], 200, "giá trị lớn nhất có thể: |-100-100|")
check_all([7, 7, 7, 7], 0, "toàn số giống nhau")
check_all([1, 2, 3, 4, 5], 4, "tăng dần: cặp vòng (5,1) cho 4")
check_all([5, 4, 3, 2, 1], 4, "giảm dần: cặp vòng (1,5) cho 4")
check_all([1, 9, 1, 9], 8, "xen kẽ: mọi cặp kề đều 8")
check_all([0, 0, 0, 50], 50, "cặp lớn nhất nằm ở cuối (0,50) và vòng (50,0)")
check_all([50, 0, 0, 0], 50, "cặp lớn nhất ở đầu")
check_all([0, 3, 0, 3, 0, 100], 100, "cặp (0,100) và (100,0)")

PASS hai phần tử bằng nhau
PASS hai phần tử: |1-100|
PASS giá trị lớn nhất có thể: |-100-100|
PASS toàn số giống nhau
PASS tăng dần: cặp vòng (5,1) cho 4
PASS giảm dần: cặp vòng (1,5) cho 4
PASS xen kẽ: mọi cặp kề đều 8
PASS cặp lớn nhất nằm ở cuối (0,50) và vòng (50,0)
PASS cặp lớn nhất ở đầu
PASS cặp (0,100) và (100,0)


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách sai thường gặp: (a) **quên cặp vòng** (`range(n - 1)`); (b) trả về **`max(nums) - min(nums)`** (độ chênh toàn mảng, không phải giữa hai phần tử **kề nhau**).

In [24]:
def wrong_no_wrap(nums):
    return max(abs(nums[i] - nums[i + 1]) for i in range(len(nums) - 1))       # SAI: thiếu cặp vòng


def wrong_max_minus_min(nums):
    return max(nums) - min(nums)                                               # SAI: không xét tính kề nhau


print(wrong_no_wrap([1, 2, 4]), "!=", sol.maxAdjacentDistance([1, 2, 4]))
print(wrong_max_minus_min([1, 5, 9, 5]), "!=", sol.maxAdjacentDistance([1, 5, 9, 5]))
assert wrong_no_wrap([1, 2, 4]) != sol.maxAdjacentDistance([1, 2, 4])
assert wrong_max_minus_min([1, 5, 9, 5]) != sol.maxAdjacentDistance([1, 5, 9, 5])

2 != 3
8 != 4


### 12.3. Đối chiếu ngẫu nhiên (có seed) và tính chất

Sinh `nums` ngẫu nhiên với `n` trong `[2, 100]`, giá trị trong `[-100, 100]` (đúng ràng buộc). Kiểm tra: (1) mọi cách cho cùng kết quả với `oracle_circular_distance`; (2) đáp án luôn nằm trong `[0, max(nums) - min(nums)]`; (3) **xoay vòng** mảng đi bất kỳ bước nào **không** đổi đáp án; (4) **đảo ngược** mảng cũng không đổi đáp án; (5) **cộng cùng một số** vào mọi phần tử không đổi đáp án (hiệu không đổi).

In [25]:
rng = random.Random(3423)
for trial in range(2000):
    n = rng.randint(2, 100)
    nums = [rng.randint(-100, 100) for _ in range(n)]
    ref = oracle_circular_distance(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
    assert 0 <= ref <= max(nums) - min(nums)
    r = rng.randrange(n)
    assert sol.maxAdjacentDistance(nums[r:] + nums[:r]) == ref, "xoay vòng không đổi đáp án"
    assert sol.maxAdjacentDistance(nums[::-1]) == ref, "đảo ngược không đổi đáp án"
    c = rng.randint(-5, 5)
    assert sol.maxAdjacentDistance([x + c for x in nums]) == ref, "dịch chuyển đều không đổi đáp án"
print("PASS 2000 mảng ngẫu nhiên đúng ràng buộc (kể cả các tính chất bất biến)")

PASS 2000 mảng ngẫu nhiên đúng ràng buộc (kể cả các tính chất bất biến)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên cặp vòng (chỉ chạy đến `n - 2`)

In [26]:
def sai_quen_vong(nums):
    best = 0
    for i in range(len(nums) - 1):                 # SAI: bỏ cặp (nums[-1], nums[0])
        best = max(best, abs(nums[i] - nums[i + 1]))
    return best


print("sai:", sai_quen_vong([1, 2, 4]), "| đúng:", Solution().maxAdjacentDistance([1, 2, 4]))

sai: 2 | đúng: 3


**Vì sao sai:** mảng vòng có `n` cặp kề, không phải `n - 1`. Ví dụ 1 là minh họa: đáp án `3` đến từ cặp `(4, 1)`. **Cách sửa:** thêm cặp vòng bằng `(i + 1) % n`, chỉ số âm, hoặc so thêm `abs(nums[-1] - nums[0])` sau vòng lặp.

### Lỗi 2: `IndexError` vì truy cập `nums[i + 1]` ở phần tử cuối

In [27]:
def sai_index_error(nums):
    best = 0
    for i in range(len(nums)):
        best = max(best, abs(nums[i] - nums[i + 1]))     # SAI: i = n - 1 thì i + 1 = n vượt giới hạn
    return best


try:
    sai_index_error([1, 2, 4])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** chỉ số hợp lệ là `0..n-1`; với `i = n - 1`, `nums[i + 1]` là `nums[n]` — không tồn tại. **Cách sửa:** `nums[(i + 1) % n]`.

### Lỗi 3: Quên `abs`

In [28]:
def sai_quen_abs(nums):
    n = len(nums)
    return max(nums[i] - nums[(i + 1) % n] for i in range(n))          # SAI: hiệu có dấu


print("sai:", sai_quen_abs([1, 2, 4]), "| đúng:", Solution().maxAdjacentDistance([1, 2, 4]))
print("sai:", sai_quen_abs([-5, -10, -5]), "| đúng:", Solution().maxAdjacentDistance([-5, -10, -5]))

sai: 3 | đúng: 3
sai: 5 | đúng: 5


**Vì sao sai:** đề hỏi **hiệu tuyệt đối**. Không có `abs`, hiệu có thể âm (ví dụ `1 - 2 = -1`) và `max` sẽ chọn một hiệu nhỏ hơn thực tế. **Cách sửa:** luôn bọc `abs(...)` quanh hiệu.

### Lỗi 4: Nhầm với `max(nums) - min(nums)`

In [29]:
nums = [1, 9, 5, 9, 1]
print("max - min          :", max(nums) - min(nums))             # 8, không phải hiệu của hai phần tử kề
print("max hiệu kề (đúng):", Solution().maxAdjacentDistance(nums))
print("các cặp kề:", [(nums[i], nums[(i + 1) % len(nums)]) for i in range(len(nums))])

max - min          : 8
max hiệu kề (đúng): 8
các cặp kề: [(1, 9), (9, 5), (5, 9), (9, 1), (1, 1)]


**Vì sao sai:** đề chỉ so **hai phần tử kề nhau**. Ở đây `1` và `9` kề nhau nên cùng cho `8`; nhưng với mảng như `[1, 5, 9, 5]` thì `max - min = 8` còn hiệu lớn nhất giữa hai phần tử kề chỉ là `4`. **Cách sửa:** chỉ so các cặp kề.

In [30]:
nums = [1, 5, 9, 5]
print("max - min:", max(nums) - min(nums), "| đáp án đúng:", Solution().maxAdjacentDistance(nums))
assert Solution().maxAdjacentDistance(nums) == 4

max - min: 8 | đáp án đúng: 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy chữ "**circular / vòng / quanh**" ⇒ nghĩ ngay đến `(i + 1) % n` hoặc chỉ số âm `nums[i - 1]`.
- Mảng vòng `n` phần tử có **`n` cặp kề** (mảng thường chỉ có `n - 1`).
- "**Hiệu tuyệt đối lớn nhất giữa các phần tử kề**" ⇒ một lượt duyệt `O(n)` với `max(abs(...))`; không cần sắp xếp.
- `itertools.pairwise(nums + [nums[0]])` là cách ngắn nhất, nhưng hãy chắc bạn hiểu nó tạo ra cặp nào.
- Cẩn thận phân biệt "**kề nhau**" (adjacent) với "**bất kỳ hai phần tử**": nếu hỏi mọi cặp thì mới là `max - min`.
- Tự kiểm tra bằng **tính chất bất biến**: xoay vòng hoặc đảo ngược mảng không đổi đáp án; cộng đều một số vào mọi phần tử không đổi đáp án.

## 15. Biến thể của bài

1. **Hiệu tuyệt đối nhỏ nhất** giữa hai phần tử kề trong mảng vòng.
2. **Trả về chỉ số** `i` của cặp `(i, (i + 1) % n)` cho hiệu lớn nhất (nếu có nhiều thì lấy `i` nhỏ nhất).
3. **Mảng thường (không vòng):** bỏ cặp vòng — bài "Maximum Gap" (LeetCode 164) là phiên bản khó hơn khi sắp xếp trước.
4. **Tổng** các hiệu tuyệt đối giữa các cặp kề trong mảng vòng.
5. **Cách nhau `k` ô** trong mảng vòng: cặp `(nums[i], nums[(i + k) % n])`. Cũng liên quan đến các bài mảng vòng như Circular Array Loop (LeetCode 457) và Maximum Sum Circular Subarray (LeetCode 918).

In [31]:
def min_adjacent_distance(nums):
    n = len(nums)
    return min(abs(nums[i] - nums[(i + 1) % n]) for i in range(n))


def argmax_adjacent(nums):
    n = len(nums)
    diffs = [abs(nums[i] - nums[(i + 1) % n]) for i in range(n)]
    return diffs.index(max(diffs))            # index() trả về vị trí đầu tiên -> i nhỏ nhất


def sum_adjacent_circular(nums):
    n = len(nums)
    return sum(abs(nums[i] - nums[(i + 1) % n]) for i in range(n))


def max_distance_k_apart(nums, k):
    n = len(nums)
    return max(abs(nums[i] - nums[(i + k) % n]) for i in range(n))


nums = [1, 2, 4]
print(min_adjacent_distance(nums))            # min(1, 2, 3) = 1
print(argmax_adjacent(nums))                  # cặp (nums[2], nums[0]) -> i = 2
print(sum_adjacent_circular(nums))            # 1 + 2 + 3 = 6
print(max_distance_k_apart(nums, 1))          # k = 1 là đề gốc -> 3
assert min_adjacent_distance(nums) == 1
assert argmax_adjacent(nums) == 2
assert sum_adjacent_circular(nums) == 6
assert max_distance_k_apart(nums, 1) == Solution().maxAdjacentDistance(nums)

1
2
6
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp + lọc kề | `j - i == 1` hoặc `n - 1` | `O(n²)` | `O(1)` | bám sát định nghĩa, làm trọng tài kiểm tra |
| 2. `(i + 1) % n` | quay vòng chỉ số | `O(n)` | `O(1)` | **nên nộp**; dùng lại cho mọi bài mảng vòng |
| 3. `nums[i - 1]` | chỉ số âm của Python | `O(n)` | `O(1)` | viết ngắn nhất trong Python |
| 4. `nums + [nums[0]]` + `pairwise` | biến cặp vòng thành cặp thường | `O(n)` | `O(n)` | gọn, đúng lời giải mẫu |
| 5. Quét `n − 1` cặp + cặp vòng | tách riêng cặp vòng | `O(n)` | `O(1)` | không dùng `%`, dễ chuyển sang C++/Java |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(2 + 1) % 3`, `(-1) % 5`, `[10, 20, 30][-1]`, `Solution().maxAdjacentDistance([3, 3])`, `Solution().maxAdjacentDistance([10, 0, 10])`.

**Bài 2 (tính tay).** Với `nums = [4, -2, 7, 1]`: liệt kê **bốn cặp kề** (kể cả cặp vòng), tính `|hiệu|` của từng cặp và tìm `max`.

**Bài 3 (điền TODO).** Hoàn thành `max_adjacent_todo(nums)` theo Cách 2 (`(i + 1) % n`).

In [32]:
def max_adjacent_todo(nums):
    n = len(nums)
    best = 0
    # TODO 1: for i in range(n)
    #   TODO 2: j = chỉ số kề sau của i, nhớ quay vòng bằng (i + 1) % n
    #   TODO 3: best = max(best, abs(nums[i] - nums[j]))
    # TODO 4: return best
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_adjacent_from_scratch(nums)` theo **Cách 5** (quét `n − 1` cặp rồi so cặp vòng) **mà không dùng** `max()` — chỉ dùng `if` để cập nhật `best`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `circular_pairs(nums)` trả về **danh sách các cặp kề** `[(nums[0], nums[1]), ..., (nums[-1], nums[0])]` bằng `zip` và cắt lát. (b) Viết `count_pairs_with_diff(nums, d)` đếm **số cặp kề** (trong mảng vòng) có `|hiệu| >= d`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(2 + 1) % 3 = 0`; `(-1) % 5 = 4` (Python luôn cho phần dư không âm khi chia cho số dương); `30`; `0`; `10`. Bài 3: dùng đúng `(i + 1) % n` cho `j`.
- **Mức 2 (vừa):** Bài 2: cặp `(4, -2) → 6`, `(-2, 7) → 9`, `(7, 1) → 6`, `(1, 4) → 3` ⇒ `max = 9`. Bài 4: `d = abs(nums[i] - nums[i + 1]); if d > best: best = d`.
- **Mức 3 (sâu):** Bài 5(a): `list(zip(nums, nums[1:] + nums[:1]))`. 5(b): `sum(1 for a, b in circular_pairs(nums) if abs(a - b) >= d)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert (2 + 1) % 3 == 0
assert (-1) % 5 == 4
assert [10, 20, 30][-1] == 30
assert Solution().maxAdjacentDistance([3, 3]) == 0
assert Solution().maxAdjacentDistance([10, 0, 10]) == 10

# Bài 2
assert Solution().maxAdjacentDistance([4, -2, 7, 1]) == 9


# Bài 3
def max_adjacent_todo_ans(nums):
    n = len(nums)
    best = 0
    for i in range(n):
        j = (i + 1) % n
        best = max(best, abs(nums[i] - nums[j]))
    return best


for nums, expected in examples:
    assert max_adjacent_todo_ans(nums) == expected


# Bài 4
def max_adjacent_from_scratch(nums):
    best = 0
    for i in range(len(nums) - 1):
        d = abs(nums[i] - nums[i + 1])
        if d > best:
            best = d
    d = abs(nums[-1] - nums[0])
    if d > best:
        best = d
    return best


for nums, expected in examples:
    assert max_adjacent_from_scratch(nums) == expected


# Bài 5
def circular_pairs(nums):
    return list(zip(nums, nums[1:] + nums[:1]))


def count_pairs_with_diff(nums, d):
    return sum(1 for a, b in circular_pairs(nums) if abs(a - b) >= d)


assert circular_pairs([1, 2, 4]) == [(1, 2), (2, 4), (4, 1)]
assert count_pairs_with_diff([1, 2, 4], 2) == 2          # |2-4| = 2 và |4-1| = 3
assert count_pairs_with_diff([1, 2, 4], 4) == 0
print("Tất cả đáp án đều đúng")

Tất cả đáp án đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hiệu lớn nhất giữa các phần tử kề của mảng vòng)

```python
class Solution:
    def maxAdjacentDistance(self, nums: List[int]) -> int:
        n = len(nums)
        best = 0
        for i in range(n):
            best = max(best, abs(nums[i] - nums[(i + 1) % n]))   # (i + 1) % n: quay vòng về đầu
        return best
```

**Checklist**

- [ ] Có xét **cặp vòng** (phần tử cuối kề phần tử đầu)?
- [ ] Không gây `IndexError` (dùng `% n`, chỉ số âm hoặc xử lý riêng cặp cuối)?
- [ ] Có `abs` quanh hiệu?
- [ ] Không nhầm với `max(nums) - min(nums)` (chỉ xét hai phần tử **kề nhau**)?
- [ ] Đã thử `n = 2`, mọi phần tử bằng nhau, và trường hợp đáp án nằm ở **cặp vòng**?
- [ ] Trả về một số nguyên `int`?